In [ ]:
import tensorflow as tf
import numpy as np

#1. Tensorflow를 통한 MNIST 다운로드
(x_train,y_train), (x_test,y_test)= tf.keras.datasets.mnist.load_data()
#2. 데이터 형태 변환 및 정규화
x_train=x_train.reshape(-1,1,28,28).astype(np.float32)
x_test=x_test.reshape(-1,1,28,28).astype(np.float32)
x_train/=255.0
x_test/=255.0

#3 정답지를 one-hot encoding으로 바꾸기
def one_hot_encoding(y,num_classes=10):
    return np.eye(num_classes)[y]

y_train_encoding=one_hot_encoding(y_train)
y_test_encoding=one_hot_encoding(y_test)


In [ ]:
import numpy as np

class ReLU:
    def __init__(self):
        self.mask=None

    def forward(self,x):
        self.mask=(x<=0)
        out=x.copy()
        out[self.mask]=0
        return out
    def backward(self,dout):
        dout[self.mask]=0
        dx=dout
        return dx


In [ ]:
class FullyConnected:
    def __init__(self,W,b):
        self.W=W
        self.b=b

        self.x=None #순전파 시의 입력값을 저장할 변수
        self.dW=None
        self.db=None

    def forward(self,x):
        self.x=x
        out=np.dot(self.x,self.W)+self.b
        return out
    def backward(self,dout):
        dx=np.dot(dout,self.W.t)
        self.dW=np.dot(self.x.T,dout)
        self.db=np.sum(dout,axis=0)

        return dx

In [ ]:
def softmax(x):
    if x.ndim==2:
        x=x.T
        x=x-np.max(x,axis=0)
        y= np.exp(x)/np.sum(np.exp(x),axis=0)  
        return y.T
    x=x-np.max(x)
    return np.exp(x)/np.sum(np.exp(x))

def cross_entropy_error(y,t):
    if y.ndim==1:
        t=t.reshape(1,t.size)
        y=y.reshape(1,y.size)
    if t.size==y.size:
        t=t.argmax(axis=1)
    batch_size=y.shape[0]
    return -np.sum(np.log(y[np.arange(batch_size),t]+1e-7))/batch_size


In [ ]:
class SoftmaxWithLoss:
    def __init__(self):
        self.loss=None
        self.y=None
        self.t=None 
    def forward(self,x,t):
        self.t=t
        self.y=softmax(x)
        self.loss=cross_entropy_error(self.y,self.t)
        return self.loss
    def backward(self,dout=1):
        batch_size=self.t.shape[0]
        dx=(self.y-self.t)/batch_size
        return dx

In [ ]:
def im2col(input_data,filter_h,filter_w,stride=1,pad=0):
    N,C,H,W=input_data.shape 
    out_h=(H+2*pad-filter_h)//stride+1
    out_w=(W+2*pad-filter_w)//stride+1
    img=np.pad(input_data,[(0,0),(0,0),(pad,pad),(pad,pad)],'constant')
    col=np.zeros(N,C,filter_h,filter_w,out_h,out_w)

    for y in range(filter_h):
        y_max=y+stride*out_h
        for x in range(filter_w):
            x_max=x+stride*out_w
            col[:,:,y,x,:,:]=img[:,:,y:y_max:stride,x:x_max:stride]
    col=col.transpose(0,4,5,1,2,3).reshape(N*out_h*out_w,-1)
    return col

In [ ]:
class Convolution:
    def __init__(self,W,b,stride=1,pad=0):
        self.W=W
        self.b=b
        self.stride=stride
        self.pad=pad

        self.x=None
        self.col=None
        self.col_W=None
        self.dW=None 
        self.db=None

    def forward(self,x):
        FN,C,FH,FW=self.W.shape
        N,C,H,W=x.shape

        out_h=int(1+(H+2*self.pad-FH)/self.stride)
        out_w=int(1+(W+2*self.pad-FW)/self.stride)

        col=im2col(x,FH,FW,self.stride,self.pad)

        col_W=self.W.reshape(FN,-1).T 
        out=np.dot(col,col_W)+self.b
        out=out.reshape(N,out_h,out_w,-1).transpose(0,3,1,2)

        self.x=x
        self.col=col
        self.col_W=col_W

        return out